# Logging 结构化日志教程 (v0.7)

本教程帮助你理解可观测性中**结构化日志**的概念和使用方式。

## 学习目标

1. 理解结构化日志：带上下文的机器可读日志
2. 掌握日志查询：按级别、模块、时间范围过滤
3. 学会上下文绑定：BoundLogger 自动附加上下文
4. 了解文件日志持久化

In [ ]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

## 1. 什么是结构化日志？

传统日志是纯文本，难以机器解析。结构化日志是**带上下文的键值对**，便于查询和关联。

| 特性 | 传统日志 | 结构化日志 |
|------|---------|----------|
| 格式 | 纯文本 | JSON/字典 |
| 查询 | grep | 按字段过滤 |
| 关联 | 手动搜索 | trace_id 关联 |
| 机器可读 | 困难 | 容易 |

In [ ]:
# 导入 Logging 相关类型
from rogue_rabbit.contracts.log import LogLevel, LogEntry
from rogue_rabbit.core.log_manager import StructuredLogger
from rogue_rabbit.runtime.log_store import InMemoryLogStore, FileLogStore

# 创建结构化日志器
store = InMemoryLogStore()
log = StructuredLogger(store=store, module="demo")

print("导入成功！")

## 2. 日志级别

四个级别，从低到高：DEBUG → INFO → WARNING → ERROR

In [ ]:
# 四个日志级别
log.debug("调试信息，只在开发时使用")
log.info("用户登录", user_id="user1", ip="192.168.1.1")
log.warning("磁盘空间不足", usage="85%")
log.error("数据库连接失败", host="localhost", port=5432)

# 查看日志条目
entries = store.query(limit=10)
print(f"日志条目数: {len(entries)}\n")
for entry in entries:
    ctx = f" | context={entry.context}" if entry.context else ""
    print(f"  [{entry.level.value:7s}] {entry.module}: {entry.message}{ctx}")

## 3. 结构化上下文

每条日志都可以携带**结构化上下文**（键值对），便于后续查询和分析。

In [ ]:
# 带丰富上下文的日志
log.info(
    "LLM 调用完成",
    model="gpt-4",
    tokens=150,
    duration_ms=230,
    status="success",
)

log.error(
    "工具调用失败",
    tool="file_reader",
    error_type="PermissionDenied",
    path="/etc/passwd",
    retry_count=3,
)

# 展示结构化内容
recent = store.query(limit=2)
for entry in recent:
    print(f"\n消息: {entry.message}")
    print(f"级别: {entry.level.value}")
    print(f"上下文:")
    for k, v in entry.context.items():
        print(f"  {k}: {v}")

## 4. 上下文绑定 (BoundLogger)

使用 `with_context()` 创建预绑定上下文的子 Logger，避免重复传参。

In [ ]:
# 清空之前的日志
store.clear()

# 创建绑定 session_id 的子 Logger
session_log = log.with_context(session_id="sess_abc123", user_id="user1")

session_log.info("会话开始")
session_log.info("处理用户输入", input_length=42)
session_log.warning("上下文窗口接近上限", usage="90%")
session_log.info("会话结束")

# 所有日志都自动携带 session_id 和 user_id
for entry in store.query():
    sid = entry.context.get("session_id", "N/A")
    uid = entry.context.get("user_id", "N/A")
    print(f"  [{entry.level.value:7s}] {entry.message} | session={sid} user={uid}")

## 5. 日志查询

支持按级别、模块、时间范围过滤。

In [ ]:
# 按级别统计
print("按级别统计:")
for level in LogLevel:
    count = store.count(level=level)
    print(f"  {level.value:7s}: {count}")

# 按级别查询
errors = store.query(level=LogLevel.WARNING)
print(f"\nWARNING 以上日志:")
for entry in errors:
    print(f"  {entry.message}")

## 6. 文件日志持久化

使用 `FileLogStore` 将日志持久化到文件系统（JSONL 格式）。

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    log_dir = Path(tmpdir) / "logs"
    
    # 写入日志
    file_store = FileLogStore(log_dir)
    file_log = StructuredLogger(store=file_store, module="app")
    
    file_log.info("服务启动", version="0.7.0")
    file_log.info("处理请求", endpoint="/api/chat", method="POST")
    file_log.error("请求超时", endpoint="/api/chat", timeout=30)
    
    # 查看文件内容
    log_files = list(log_dir.glob("*.log"))
    print(f"日志文件: {[f.name for f in log_files]}")
    
    # 重新加载查询
    store2 = FileLogStore(log_dir)
    entries = store2.query()
    print(f"\n重新加载后日志数: {len(entries)}")
    for entry in entries:
        print(f"  [{entry.level.value:7s}] {entry.message}")

## 总结

### 核心概念

| 概念 | 说明 |
|------|------|
| `LogEntry` | 单条结构化日志（级别 + 消息 + 模块 + 上下文） |
| `StructuredLogger` | 日志写入器，支持四个级别 |
| `BoundLogger` | 预绑定上下文，避免重复传参 |
| `LogStore` | 日志存储后端（InMemory/File） |

### 下一步

- 运行 `experiments/18_logging_basic.py`
- 继续学习 [09_tracing.ipynb](09_tracing.ipynb) 请求追踪